# 🚀 Blucher BuildTrack - Google Colab GPU AI Brain (Ollama + Qwen2.5)
## สถาปัตยกรรม Hybrid AI Brain สำหรับระบบติดตามโครงการ บลูเคอร์ (Blucher)

สมุดงานนี้รันโมเดล AI ภาษาไทย (LLM) บน **Google Colab Free GPU (T4 / L4)** 100% ฟรี

### 🛡️ กฎเหล็กความปลอดภัยและการประหยัดทรัพยากร:
- **0% Local Load:** ไม่ใช้ทรัพยากร CPU/GPU/RAM บนคอมพิวเตอร์ของพี่นัทเลยแม้แต่น้อย ทำงานบน Cloud GPU ของ Google ทั้งหมด
- **Hybrid Architecture:** ทำงานร่วมกับ **Google Cloud Gemini 2.5 Flash** เมื่อ Colab ทำงาน ระบบจะแบ่งเบางาน Text Inference และ Q&A ทันที
- **Automatic Tunnel Pairing:** สคริปต์ด้านล่างจะเปิด Cloudflare Secure Tunnel และส่ง Tunnel URL ไปยัง Cloudflare Worker ของ Blucher อัตโนมัติโดยที่พี่นัทไม่ต้องก็อปปี้วางเอง!

### 📝 ขั้นตอนการใช้งานง่ายๆ:
1. ตรวจสอบว่า Runtime Type เป็น **T4 GPU** (เมนู `Runtime` > `Change runtime type` > เลือก `T4 GPU`)
2. กดปุ่ม **Runtime > Run all** (หรือกดปุ่ม Play ที่เซลล์ด้านล่างทีละเซลล์)
3. เมื่อเซลล์ที่ 2 ขึ้นข้อความ `✅ ส่ง Tunnel URL ไปยัง Blucher Project Tracker สำเร็จ!` ระบบจะเชื่อมต่อกับบอทและหน้าเว็บทันที!

In [ ]:
# ==============================================================================
# Cell 2: ติดตั้งและเปิดเซิร์ฟเวอร์ Ollama LLM + Cloudflared Tunnel บน Colab GPU
# ==============================================================================
import os
import subprocess
import time
import requests
import re

print("🚀 [Step 1/5] ตรวจสอบ GPU บน Google Colab...")
!nvidia-smi

print("\n📦 [Step 2/5] ติดตั้ง Ollama Server บนระบบ Linux...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\n⚡ [Step 3/5] เริ่มทำงาน Ollama Daemon ใน Background...")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

print("\n🧠 [Step 4/5] ดาวน์โหลดโมเดลภาษาไทยประสิทธิภาพสูง (Qwen2.5:7b)... ")
!ollama pull qwen2.5:7b

print("\n🌐 [Step 5/5] ติดตั้งและสร้าง Cloudflare Secure Tunnel...")
!curl -fsSL -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# รัน Cloudflared Tunnel ไปยัง localhost:11434 (พอร์ตของ Ollama)
tunnel_log_file = "tunnel.log"
if os.path.exists(tunnel_log_file):
    os.remove(tunnel_log_file)

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=open(tunnel_log_file, "w"),
    stderr=subprocess.STDOUT
)

print("⏳ กำลังรอ Tunnel URL จาก Cloudflare...")
tunnel_url = None
for i in range(30):
    time.sleep(1)
    if os.path.exists(tunnel_log_file):
        with open(tunnel_log_file, "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print(f"\n🎉 สำเร็จ! ได้รับ Colab GPU Tunnel URL: {tunnel_url}")
    WORKER_ENDPOINT = "https://blucher-project-tracker.natt-charoen.workers.dev/api/settings/colab-brain"
    try:
        res = requests.post(WORKER_ENDPOINT, json={"colabUrl": tunnel_url}, timeout=10)
        if res.status_code == 200:
            print("✅ ส่ง Tunnel URL ไปยัง Blucher Project Tracker สำเร็จ!")
            print(f"📊 ผลตอบกลับจาก Worker: {res.json()}")
        else:
            print(f"⚠️ Worker ตอบกลับรหัส: {res.status_code}")
    except Exception as e:
        print(f"⚠️ ไม่สามารถส่ง Tunnel URL อัตโนมัติ: {e}")
        print(f"👉 สามารถคัดลอก URL นี้ไปวางใน Dashboard ได้ด้วยตนเอง: {tunnel_url}")
else:
    print("❌ ไม่พบ Tunnel URL กรุณาตรวจสอบ tunnel.log")

In [ ]:
# ==============================================================================
# Cell 3: Keep-Alive Heartbeat (รักษาการเชื่อมต่อและป้องกัน Colab หลุด)
# ==============================================================================
import time
import requests

print("💓 เริ่มต้นระบบ Keep-Alive Heartbeat สำหรับ Blucher Colab Brain...")
print("💡 เปิดแท็บนี้ทิ้งไว้เพื่อให้ระบบทำงานอย่างต่อเนื่อง (กด Stop เมื่อต้องการปิดการทำงาน)")

count = 1
while True:
    try:
        res = requests.get("http://localhost:11434/api/tags", timeout=5)
        status = "Active" if res.status_code == 200 else f"Err {res.status_code}"
        print(f"[{time.strftime('%H:%M:%S')}] Heartbeat #{count} - GPU & Ollama: {status}")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] Heartbeat #{count} - Warning: {e}")
    count += 1
    time.sleep(300)